# Import Libraries

In [1]:
import os
import zipfile
import requests
from pathlib import Path

import torch
from torchvision import transforms

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
data_path = Path("m_data")
image_path = data_path / "pizza_steak_sushi"

# Check if the image path exists or not.
if image_path.is_dir():
    print(f"{image_path} directory already exists, skipping download.")
else:
    print(f"{image_path} directory does not exist, creating one and downloading data.")
    image_path.mkdir(parents=True, exist_ok=True)
    
    # Download the data
    with requests.get(
        "https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip"
    ) as r:
        with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
            f.write(r.content)      
    # Unzip the data
    with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
        zip_ref.extractall(image_path),
    # Remove the zip file
    os.remove(data_path / "pizza_steak_sushi.zip")  

m_data/pizza_steak_sushi directory already exists, skipping download.


# 2. Preprocessing 

In [4]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [5]:
# image paths for train and test datas.
train_dir = image_path / "train"
test_dir = image_path / "test"
Batch_size = 32
num_workers = os.cpu_count()

In [6]:
# import the module
from going_modular.data_setup import create_dataloaders

train_dataloader, test_dataloader, class_names = create_dataloaders(train_dir=train_dir,
                                                                    test_dir=test_dir,
                                                                    transform=transform,
                                                                    batch_size=Batch_size,
                                                                    num_workers=num_workers)

Class names: ['pizza', 'steak', 'sushi']


# 3. Define TinyVGG model Architecture

In [7]:
from going_modular.model_builder import TinyVGG

torch.manual_seed(42)
model = TinyVGG(input_shape=3,
                 hidden_units=10,
                 output_shape=len(class_names)).to(device)

In [8]:
print(model)

TinyVGG(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (Classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1690, out_features=3, bias=True)
  )
)


# 4. Training Loop

In [9]:
# now i can import the training loop from engin module
from going_modular.engine import train_loop

# 5. Save Model

In [10]:
#  import moduls
from going_modular.utils import save_model

# 6. Train Model

In [11]:
import main

#  path(__file__) for script and path().cwd() for notebook
 # Setup directories
train_dir = "m_data/pizza_steak_sushi/train"
test_dir = "m_data/pizza_steak_sushi/test"
model_dir = Path().cwd()/"model"       # This points to the current working directory and then to the model directory
main.main(train_dir=train_dir, 
          test_dir=test_dir, 
          save_dir=model_dir)

Class names: ['pizza', 'steak', 'sushi']


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.0991 | train_acc: 0.2617 | test_loss: 1.1118 | test_acc: 0.2396
Epoch: 2 | train_loss: 1.0953 | train_acc: 0.3086 | test_loss: 1.1178 | test_acc: 0.1979
Epoch: 3 | train_loss: 1.0891 | train_acc: 0.2930 | test_loss: 1.1218 | test_acc: 0.2500
Epoch: 4 | train_loss: 1.1016 | train_acc: 0.4492 | test_loss: 1.1181 | test_acc: 0.3229
Epoch: 5 | train_loss: 1.0620 | train_acc: 0.5703 | test_loss: 1.0866 | test_acc: 0.2917
Saving the model to: /home/nigatie/Documents/ML_DL/Deep_Learning/Modularity/model/05_going_modular_script_mode_tinyvgg_model.pth
Model saved successfully!
